# 02 Run an agent with read-only tools
Let Gemini choose tools in a reason -> act -> observe loop. This agent can search policy, look up orders, and draft text, but has no refund tool. All orders and refunds are fictional, and the handbook search only reads Session 10's `heading-800` namespace. Create `.env` with both keys before using Google or Pinecone; local order/refund checks can run without them.

Find the repository whether Jupyter starts here or in the notebooks folder. Keep the two Google model names visible, and import the shared implementation so all notebooks use the same tool behavior.

In [ ]:
import os
import sys
import logging
from pathlib import Path
from dotenv import load_dotenv
EMBEDDING_MODEL = "google_genai:gemini-embedding-001"
CHAT_MODEL = "google_genai:gemini-3.7-flash"
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import agent as lab
load_dotenv(ROOT / ".env")

Check that the required files and key names are present without printing secrets. The log labels show observable tool choices and results, not hidden model reasoning; LangSmith tracing is disabled by the shared module.

In [ ]:
logging.basicConfig(level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s", force=True)
missing = []
for name in ("GOOGLE_API_KEY", "PINECONE_API_KEY"):
    if not os.getenv(name):
        missing.append(name)
if missing:
    print("Add these keys to .env:", ", ".join(missing))
else:
    print("Both keys are present; live requests check access.")
assert lab.ORDERS_PATH.is_file(), "Restore data/orders.csv before continuing."

Create the agent with the exact system instruction and the three read-only tools. A checkpointer lets the shared trace runner inspect state; it is only in memory and makes no external logging connection.

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
read_only_tools = [lab.search_handbook, lab.lookup_order, lab.draft_reply]
agent = create_agent(
    model=lab.get_model(), tools=read_only_tools,
    system_prompt=lab.SYSTEM_PROMPT, checkpointer=InMemorySaver(),
)

Run the worn-boots ticket and follow the REASON, ACT, OBSERVE, and DONE logs. The runner uses `recursion_limit=10`, counts model/tool updates, and prints the full messages with `pretty_print()` so you can inspect citations and tool arguments.

In [ ]:
message = "Can I return boots I wore on one hike? Order T-1007."
result = lab.run_agent(agent, message)
print("Model/tool updates:", result["steps"])
print("Stopped early:", result.get("stopped", False))

Inspect the final reply separately from the complete trace above. Check the footwear exception and `[Policies > Returns]`; a tool calling loop still needs human review for answer quality.

In [ ]:
if result["messages"]:
    print(result["messages"][-1].content)

## See the hard stop
Use an intentionally tiny recursion limit so the graph has no room to complete the policy/order/reply sequence. Catching `GraphRecursionError` turns that limit into a clear stop message rather than an uncontrolled loop.

In [ ]:
from langgraph.errors import GraphRecursionError
from uuid import uuid4
try:
    agent.invoke({"messages": [{"role": "user", "content": message}]},
        config={"recursion_limit": 1, "configurable": {"thread_id": str(uuid4())}})
except GraphRecursionError:
    print("Agent stopped: step limit reached.")
except Exception:
    print("Request failed: check keys; for 429/500, wait and retry.")